# RainGuard — Module 3: Statistical Analysis
**Indian Rainfall Forecasting and Heavy Rainfall Risk Analysis System**

**Input:** `data/processed/rainfall_cleaned.csv` (Module 1 output). **Output:** 5 figures (`docs/figures/m3_*.png`) and the statistical evidence that decides how Modules 4–6 are built.

**Why this module exists.** EDA *describes* the data; statistical analysis *tests* what we saw and measures how sure we are. Each step below answers one question the later modules depend on:

| Step | Question | Decision it drives |
|---|---|---|
| 2 | What do the summary statistics look like by state? | Baseline understanding |
| 3 | Is rainfall normally distributed? | Which tests and which error metrics are valid |
| 4 | Does yesterday's rain tell us about tomorrow's? | Whether lag features are worth building (Module 4) |
| 5 | Do August, September and October really differ? | Whether `month` / `day_of_year` carry signal |
| 6 | Do states really differ? | How to encode location |
| 7 | How uncertain are our headline numbers? | Confidence intervals instead of single numbers |
| 8 | Which candidate predictors relate to *tomorrow's* rain? | Feature shortlist for Module 4 |

**Reading the numbers correctly.** The data covers 19 Aug – 30 Oct 2026 (73 days). Consecutive days in one district are **not independent** (Step 4 proves it), so p-values below are indicative; we therefore always report an *effect size* next to a p-value, and use district-level summaries or a district-level bootstrap where possible.

## Step 1 — Imports, paths and chart style
Same paths and colour rules as Module 2.

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd
CLEAN_PATH = os.path.join(PROJECT_ROOT, "data", "processed", "rainfall_cleaned.csv")
FIG_DIR = os.path.join(PROJECT_ROOT, "docs", "figures")
os.makedirs(FIG_DIR, exist_ok=True)
if not os.path.exists(CLEAN_PATH):
    raise FileNotFoundError("Run the Module 1 notebook first to create " + CLEAN_PATH)

BLUE, ORANGE = "#2a78d6", "#eb6834"
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#b9b8b2", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titleweight": "bold", "axes.titlesize": 12, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7, "axes.axisbelow": True,
    "font.size": 10, "legend.frameon": False,
})

def save_show(name):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"m3_{name}.png"), dpi=150)
    plt.show()

def pretty(name):
    return name.title().replace("Nct Of", "NCT of").replace(" And ", " and ")

date_cols = ["date", "cumulative_date", "week_start", "week_end", "month_start", "month_end"]
df = pd.read_csv(CLEAN_PATH, parse_dates=date_cols)
df = df.sort_values(["state", "district", "date"]).reset_index(drop=True)
grp = df.groupby(["state", "district"])
x = df["daily_actual"]
HEAVY_MM = 64.5
print("Shape:", df.shape, "| dates:", df["date"].min().date(), "->", df["date"].max().date())

## Step 2 — Descriptive statistics
**What we are doing:** summary numbers for all records and for each state. We report the **median and the 95th percentile next to the mean**, because for zero-inflated data the mean alone is misleading.

```
Heavy-or-above = at least 64.5 mm in a day (IMD Heavy + Very Heavy + Extremely Heavy)
```

In [ ]:
overall = x.describe(percentiles=[.5, .9, .95, .99]).round(2).to_frame("daily_actual (mm)")
overall.loc["skewness"] = round(x.skew(), 2)
overall.loc["excess kurtosis"] = round(x.kurt(), 2)
overall.loc["share of 0 mm days (%)"] = round((x == 0).mean() * 100, 1)
display(overall)

state_tbl = (df.groupby("state")
               .agg(districts=("district", "nunique"),
                    mean_mm=("daily_actual", "mean"),
                    median_mm=("daily_actual", "median"),
                    std_mm=("daily_actual", "std"),
                    p95_mm=("daily_actual", lambda s: s.quantile(.95)),
                    rainy_pct=("daily_actual", lambda s: (s > 0).mean() * 100),
                    heavy_plus=("daily_actual", lambda s: int((s >= HEAVY_MM).sum())))
               .sort_values("mean_mm", ascending=False).round(2))
print("\nState-wise statistics (sorted by mean daily rainfall):")
display(state_tbl.head(10))
print("... driest 5:")
display(state_tbl.tail(5))

**Findings.**
* The national mean is **4.28 mm/day**, but the median is **0** and the standard deviation is **11.4 mm**: the average is pulled up by a minority of wet days. The 99th percentile is 55.2 mm, and the maximum 231.9 mm.
* Skewness (5.7) and excess kurtosis (≈ 51) are far from the values of a normal distribution (0 and 0), which we test formally in Step 3.
* For every state the standard deviation is much larger than the mean, which is the bursty behaviour seen through the coefficient of variation in Module 2.

**Why it matters.** The mean is a poor "typical day". Reporting only a mean would hide the fact that most days are dry.

### Step 2b — Quartiles and district-wise comparison
**What we are doing:** the five-number summary (minimum, Q1, median, Q3, maximum) and the IQR (Q3 − Q1), once for *all* district-days and once for *rainy days only*, because the zeros dominate the first. Then we compare districts: the wettest and driest, and how district averages are spread.

In [ ]:
rainy = x[x > 0]
def five(s):
    return pd.Series({"min": s.min(), "Q1 (25%)": s.quantile(.25), "median": s.median(),
                      "Q3 (75%)": s.quantile(.75), "max": s.max(), "IQR": s.quantile(.75) - s.quantile(.25)})
quart = pd.DataFrame({"all district-days": five(x), "rainy days only (>0 mm)": five(rainy)}).round(2)
display(quart)

dist_tbl = (grp["daily_actual"].agg(mean_mm="mean", median_mm="median", std_mm="std", max_mm="max",
                                    days="size", total_mm="sum").reset_index())
dist_tbl["cv"] = dist_tbl["std_mm"] / dist_tbl["mean_mm"].replace(0, np.nan)
dist_tbl["rainy_pct"] = grp["daily_actual"].apply(lambda s: (s > 0).mean() * 100).values
fulld = dist_tbl[dist_tbl["days"] == df["date"].nunique()]
cols = ["state", "district", "mean_mm", "median_mm", "std_mm", "max_mm", "rainy_pct"]
print("\nDistrict-wise comparison (districts with all 73 days): wettest 5")
display(fulld.nlargest(5, "mean_mm")[cols].round(2))
print("Driest 5 (excluding the all-zero districts):")
display(fulld[fulld["mean_mm"] > 0].nsmallest(5, "mean_mm")[cols].round(2))
print("District mean daily rainfall - quartiles across 727 districts:")
display(fulld["mean_mm"].describe().round(2).to_frame("mm/day"))

**Findings.**
* **All district-days:** Q1, median and minimum are all 0 mm, and Q3 is only 2.9 mm, so three quarters of district-days have under 3 mm of rain.
* **Rainy days only:** the typical rainy day is light (median 3.9 mm, middle 50% between 1.0 and 11.6 mm), yet the maximum is 231.9 mm, about 60 times the median.
* **Wettest districts (mean mm/day):** North & Middle Andaman (20.4, rain on 85% of days), Koraput (13.2), Nabarangapur (12.6), Kolasib (12.5) and Banda (12.3). Banda and Koraput have a median of 0 to 1.9 mm but a mean above 12 mm: their rain comes in a few extreme days (maxima of 153 and 204 mm), not steadily.
* **Driest districts:** Leh & Ladakh (0.05), Jaisalmer (0.08) and Jodhpur (0.09) mm/day, with rain on only 4–8% of days.
* **Across the 727 complete districts:** the middle 50% average between 2.0 and 6.0 mm/day (median 4.0, about 290 mm over the 73 days), with a long upper tail up to 20.4.

**Why it matters.** Medians and quartiles describe a "typical" day far better than the mean here, and the gap between mean and median in a district is itself a measure of how extreme-event-driven its rainfall is.

## Step 2c — Variability and correlation
**What we are doing:**
* **Variability:** the coefficient of variation (CV = standard deviation / mean) of each district's daily rainfall, summarised per state (states with at least 5 districts, districts with 0 mm excluded).
* **Correlation:** daily rainfall against the other numeric rainfall columns, with *both* Pearson (straight-line relationship) and Spearman (rank relationship). These columns are for understanding only and will **not** be model inputs (Module 1, Step 12).

In [ ]:
v = fulld.dropna(subset=["cv"])
st_cv = (v.groupby("state").agg(districts=("district", "size"), median_cv=("cv", "median"), median_mean_mm=("mean_mm", "median"))
           .query("districts >= 5").sort_values("median_cv"))
print("National median CV of district daily rainfall:", round(v["cv"].median(), 2))
print("\nSteadiest states (lowest median CV):"); display(st_cv.head(5).round(2))
print("Most erratic states (highest median CV):"); display(st_cv.tail(5).round(2))
print("Correlation between a district's mean rainfall and its CV:",
      round(stats.spearmanr(v["mean_mm"], v["cv"])[0], 2), "(Spearman)")

others = ["daily_normal", "weekly_actual", "monthly_actual", "cumulative_actual"]
cor = pd.DataFrame({"Pearson": [x.corr(df[c]) for c in others],
                    "Spearman": [x.corr(df[c], method="spearman") for c in others]}, index=others).round(2)
display(cor)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
pos = np.arange(len(others)); w = 0.38
axes[0].bar(pos - w / 2, cor["Pearson"], w, color=BLUE, label="Pearson")
axes[0].bar(pos + w / 2, cor["Spearman"], w, color=ORANGE, label="Spearman")
axes[0].set_xticks(pos); axes[0].set_xticklabels([c.replace("_", "\n") for c in others])
axes[0].set(ylabel="Correlation with daily_actual", title="Pearson vs Spearman"); axes[0].legend()
axes[0].grid(axis="x", visible=False)
sel = pd.concat([st_cv.head(5), st_cv.tail(5)])
axes[1].barh([pretty(s) for s in sel.index], sel["median_cv"], color=BLUE, height=0.6)
axes[1].set(xlabel="Median CV of district daily rainfall", title="Steadiest and most erratic states")
axes[1].grid(axis="y", visible=False)
save_show("00_variability_correlation")

**Findings.**
* **Variability:** the median district has a CV of **2.33**, so its day-to-day standard deviation is more than twice its mean. The steadiest states are the wet ones: Mizoram (1.62), Meghalaya (1.68), Kerala (1.74), West Bengal (1.79) and Nagaland (1.89). The most erratic are the dry north-west and Delhi: **Gujarat (4.31)**, Punjab (3.59), Rajasthan (3.44), Haryana (3.16) and NCT of Delhi (3.14).
* Across districts, drier means go with higher CV (Spearman −0.57): where it rains rarely, it rains in rare bursts.
* **Pearson vs Spearman:** every rank correlation (0.41 to 0.48) is about double the Pearson one (0.19 to 0.34). The relationship between daily rainfall and the other measures is strongly monotonic but not a straight line, so Pearson understates it on this zero-inflated data. This refines the Module 2 heatmap, which used Pearson.

**Why it matters.** The most erratic regions are the hardest to forecast and the ones where a few millimetres of error are large relative to the average. From here on we prefer rank-based (Spearman) measures. Note that weekly, monthly and cumulative totals include the same rain as `daily_actual`, so part of their correlation is mechanical, one more reason they stay out of the model.

## Step 3 — Is rainfall normally distributed?
**What we are doing:** two formal tests (Shapiro–Wilk on a random sample of 5,000 values, because it is designed for samples of that size, and D'Agostino–Pearson on all values) plus a **Q–Q plot**. On a Q–Q plot, points on the diagonal mean "normal"; a curve means "not normal". We also try a `log(1 + rainfall)` transform to see whether it rescues normality.

```
H0: the data come from a normal distribution.  Reject H0 when p < 0.05.
```

In [ ]:
sample = x.sample(5000, random_state=42)
sh_stat, sh_p = stats.shapiro(sample)
dp_stat, dp_p = stats.normaltest(x)
logx = np.log1p(x)
print(f"Shapiro-Wilk (n=5000)   : W = {sh_stat:.3f}, p = {sh_p:.2e}")
print(f"D'Agostino-Pearson (all): K2 = {dp_stat:,.0f}, p = {dp_p:.2e}")
print(f"Skewness  raw: {x.skew():.2f}   after log(1+x): {logx.skew():.2f}")
print(f"Excess kurtosis raw: {x.kurt():.2f}   after log(1+x): {logx.kurt():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, data, title in [(axes[0], sample, "Raw rainfall"), (axes[1], np.log1p(sample), "After log(1 + rainfall)")]:
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist="norm")
    ax.scatter(osm, osr, s=6, color=BLUE, alpha=0.5, linewidths=0)
    ax.plot(osm, slope * np.asarray(osm) + intercept, color=ORANGE, linewidth=1.5)
    ax.set(xlabel="Theoretical normal quantiles", ylabel="Observed values", title=title)
save_show("01_normality")

**Findings.**
* Both tests reject normality with p-values that are effectively zero (Shapiro–Wilk W = 0.41; a normal sample would give W near 1).
* The Q–Q plot of raw rainfall bends sharply away from the line. After `log(1 + x)` the skewness falls from 5.7 to 1.4, but the Q–Q plot still shows a flat shelf at the bottom: that is the 55.6% of days with exactly 0 mm, which no transform can remove.

**Why it matters.**
1. We use **non-parametric tests** (Kruskal–Wallis, Friedman, Wilcoxon, Spearman) from now on, since they do not assume normality.
2. A plain linear regression on raw rainfall violates its assumptions. In Module 5 we will favour tree-based models and judge them with MAE/RMSE against a naive baseline, not with R² alone.
3. With 53,102 records, normality tests reject almost anything; the real evidence is the skewness and the Q–Q shape, not the p-value.

## Step 4 — Does rain persist from one day to the next?
**What we are doing:** this is the most important test for the project. We pair each district-day with the **same district's next day**, using only pairs that are exactly **one calendar day apart** (the rule from Module 1). Then we measure:
* the rank correlation (Spearman) between today's and tomorrow's rainfall, for lags of 1 to 7 days;
* the chance of rain tomorrow given rain today versus given a dry day, with a **chi-square test of independence**.

```
Lag k pair is valid only if date(next) - date(today) == k days
```

In [ ]:
def lagged_pairs(k):
    nxt = grp["daily_actual"].shift(-k)
    gap = (grp["date"].shift(-k) - df["date"]).dt.days
    ok = gap == k
    return x[ok], nxt[ok]

acf = []
for k in range(1, 8):
    a, b = lagged_pairs(k)
    acf.append({"lag_days": k, "pairs": len(a),
                "spearman": stats.spearmanr(a, b)[0], "pearson": stats.pearsonr(a, b)[0]})
acf = pd.DataFrame(acf).set_index("lag_days")
display(acf.round(3))

a, b = lagged_pairs(1)
rain_today, rain_next = a > 0, b > 0
table = pd.crosstab(rain_today.rename("rain today"), rain_next.rename("rain tomorrow"))
chi2, p_chi, _, _ = stats.chi2_contingency(table)
p_rr, p_dr = rain_next[rain_today].mean(), rain_next[~rain_today].mean()
print("\nRain today vs rain tomorrow (consecutive-day pairs):")
display(table)
print(f"P(rain tomorrow | rain today) = {p_rr:.1%}")
print(f"P(rain tomorrow | dry today)  = {p_dr:.1%}")
print(f"chi-square = {chi2:,.0f}, p = {p_chi:.2e}, effect size phi = {np.sqrt(chi2 / table.values.sum()):.2f}")

# is this just 'wet places stay wet'? check the correlation inside each district separately
def lag1_within(s):
    v = s.values
    return stats.spearmanr(v[:-1], v[1:])[0] if v.std() > 0 and len(v) > 5 else np.nan
within = grp["daily_actual"].apply(lag1_within)
print(f"\nLag-1 Spearman computed inside each district: median = {within.median():.2f}, "
      f"middle 80% of districts between {within.quantile(.1):.2f} and {within.quantile(.9):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].bar(acf.index.astype(str), acf["spearman"], color=BLUE, width=0.6)
for i, v in enumerate(acf["spearman"]):
    axes[0].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9, color=INK2)
axes[0].set(xlabel="Lag (days)", ylabel="Spearman correlation with rain 'lag' days later",
            title="Memory of rainfall fades with lag")
axes[0].grid(axis="x", visible=False)
axes[1].bar(["After a\ndry day", "After a\nrainy day"], [p_dr * 100, p_rr * 100], color=[ORANGE, BLUE], width=0.5)
for i, v in enumerate([p_dr * 100, p_rr * 100]):
    axes[1].text(i, v, f"{v:.0f}%", ha="center", va="bottom", fontsize=10, color=INK2)
axes[1].set(ylabel="Chance of rain the next day (%)", ylim=(0, 100), title="Rain tends to follow rain")
axes[1].grid(axis="x", visible=False)
save_show("02_persistence")

**Findings.**
* The one-day-ahead Spearman correlation is **0.57**. It drops quickly to 0.43 at lag 2 and 0.34 at lag 3, then flattens at about 0.27–0.30 for lags 4 to 7. The flat tail is the "wet places stay wet" effect (some districts are simply rainier than others), while the steep early drop is the genuine short-term memory of weather.
* After a dry day the chance of rain the next day is **20%**; after a rainy day it is **74%**. The chi-square test rejects independence with p ≈ 0, and the effect size (phi ≈ 0.53) is large, not just statistically significant.
* The effect is **not only a between-district artefact**: computed inside each district, the median lag-1 correlation is 0.52, and the middle 80% of districts lie between 0.24 and 0.71.
* The Pearson correlation is much weaker (0.41 at lag 1, below 0.1 from lag 3). That is because Pearson is dominated by the few extreme days, whereas the rank-based Spearman captures the more common pattern of "rain follows rain".

**Why it matters.**
* This is the statistical justification for **lag features** in Module 4: yesterday's rainfall is clearly informative, 2–3 days of history add something, and lags beyond about 3 days add little beyond what the district's average level already says.
* The observations are **not independent** (consecutive days are strongly related). Hence a random train/test split would leak information, and we split chronologically.
* Honest limit: a correlation of 0.57 means skill is real but modest, and it is much weaker for the extreme days (Pearson), which is exactly where we need the model.

## Step 5 — Do August, September and October really differ?
**What we are doing:** the EDA showed the mean falling from 6.5 → 5.6 → 2.0 mm/day. Here we test whether that is more than chance. Because the same districts appear in every month, we use each district's **mean daily rainfall per month** and compare the three months with the **Friedman test** (the non-parametric test for repeated measurements on the same units), followed by pairwise **Wilcoxon signed-rank** tests with a Bonferroni correction (p × 3).

**Reminder:** August is only 13 days (19–31 Aug), a partial month.

In [ ]:
dm = df.groupby(["state", "district", "month"])["daily_actual"].mean().unstack().dropna()
dm.columns = ["Aug (19-31)", "Sep", "Oct"]
n_d, k = len(dm), 3
fr_stat, fr_p = stats.friedmanchisquare(dm.iloc[:, 0], dm.iloc[:, 1], dm.iloc[:, 2])
kendall_w = fr_stat / (n_d * (k - 1))
print(f"Districts compared: {n_d}")
print("Mean of the district means (mm/day):", dm.mean().round(2).to_dict())
print(f"Friedman chi-square = {fr_stat:.1f}, p = {fr_p:.2e}, Kendall's W (effect size, 0-1) = {kendall_w:.2f}")

print("\nPairwise Wilcoxon signed-rank (Bonferroni-adjusted p):")
rows = []
for p_, q_ in [(0, 1), (1, 2), (0, 2)]:
    w = stats.wilcoxon(dm.iloc[:, p_], dm.iloc[:, q_])
    rows.append({"comparison": f"{dm.columns[p_]} vs {dm.columns[q_]}",
                 "median difference (mm/day)": (dm.iloc[:, p_] - dm.iloc[:, q_]).median(),
                 "raw p": w.pvalue, "adjusted p": min(1.0, w.pvalue * 3)})
pw = pd.DataFrame(rows)
pw["median difference (mm/day)"] = pw["median difference (mm/day)"].round(3)
print(pw.to_string(index=False, formatters={"raw p": "{:.2e}".format, "adjusted p": "{:.2e}".format}))

fig, ax = plt.subplots(figsize=(7.5, 4.3))
sns.boxplot(data=dm, color=BLUE, saturation=0.8, width=0.5, showfliers=False, linewidth=0.9, ax=ax)
ax.set(ylabel="District's mean daily rainfall in the month (mm)", title="District rainfall by month (outliers hidden)")
ax.grid(axis="x", visible=False)
save_show("03_months")

**Findings.**
* The Friedman test is highly significant (p ≈ 1e-101) with Kendall's W ≈ 0.32, a **moderate-to-large** effect: the month matters.
* The drop is concentrated in **October**: September vs October and August vs October both differ by about 3.6 mm/day (median of the district differences) and are overwhelmingly significant (adjusted p far below 1e-60, printed in scientific notation).
* **August vs September is *not* clearly different** after correcting for three comparisons (raw p = 0.027, adjusted p ≈ 0.08), and the median district difference is only −0.03 mm/day, essentially zero. The EDA's 6.5 vs 5.6 mm gap in *means* is driven by a few very wet district-days, so it should not be over-interpreted.

**Why it matters.** There is a real seasonal decline, but it is mostly one step (September → October). With only 73 days we cannot claim to learn a seasonal cycle; `day_of_year` (a smoother version of the same signal) is more useful than `month`, which has just three values. `year` stays dropped.

## Step 6 — Do states really differ?
**What we are doing:** we test whether *where* a district is predicts how much rain it gets. Two measures on the district-level means:
* **Kruskal–Wallis** test across states that have at least 5 districts;
* **eta-squared** (η²), the *share of the variation between districts that is explained by state membership*.

In [ ]:
dist_mean = grp["daily_actual"].mean().rename("mean_daily").reset_index()

big = dist_mean.groupby("state").filter(lambda d: len(d) >= 5)
kw_stat, kw_p = stats.kruskal(*[d["mean_daily"].values for _, d in big.groupby("state")])

gm = dist_mean["mean_daily"].mean()
ss_between = dist_mean.groupby("state")["mean_daily"].apply(lambda s: len(s) * (s.mean() - gm) ** 2).sum()
ss_total = ((dist_mean["mean_daily"] - gm) ** 2).sum()
eta2 = ss_between / ss_total

print(f"Kruskal-Wallis across {big['state'].nunique()} states with >= 5 districts: H = {kw_stat:.1f}, p = {kw_p:.2e}")
print(f"eta-squared: state membership explains {eta2:.0%} of the variation in district mean rainfall")
print(f"The other {1 - eta2:.0%} is variation between districts inside the same state")

**Findings.**
* States differ significantly (Kruskal–Wallis H ≈ 375 over 28 states, p ≈ 1e-62).
* **State membership explains about 51% of the variation** in district mean rainfall, so location matters a lot, but the other ≈ 49% lies *between districts of the same state* (consistent with the wide boxes in EDA Chart 5).

**Why it matters.** A state label alone is not enough; a district's own recent rainfall (lags) carries information that the state cannot. We keep the saved `state_enc`/`district_enc` for reference, but see Step 8 for an important warning about using them as numbers.

## Step 7 — Confidence intervals (how sure are our headline numbers?)
**What we are doing:** a single number such as "4.28 mm/day" has uncertainty. Records from the same district are related, so we resample **whole districts** (a *cluster bootstrap*: draw 728 districts with replacement, recompute, repeat 2,000 times). The 2.5th and 97.5th percentiles of the results give a **95% confidence interval**. This is more honest than treating all 53,102 rows as independent.

In [ ]:
agg = grp["daily_actual"].agg(total="sum", n="size")
agg["rainy"] = grp["daily_actual"].apply(lambda s: (s > 0).sum())
agg["heavy"] = grp["daily_actual"].apply(lambda s: (s >= HEAVY_MM).sum())
arr = agg[["total", "n", "rainy", "heavy"]].values.astype(float)

rng = np.random.default_rng(42)
B, n_dist = 2000, len(arr)
boot = np.empty((B, 3))
for i in range(B):
    s = arr[rng.integers(0, n_dist, n_dist)].sum(axis=0)
    boot[i] = [s[0] / s[1], s[2] / s[1] * 100, s[3] / s[1] * 100]

tot = arr.sum(axis=0)
point = np.array([tot[0] / tot[1], tot[2] / tot[1] * 100, tot[3] / tot[1] * 100])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
names = ["Mean daily rainfall (mm)", "Share of district-days with rain (%)", "Heavy-or-above district-days (%)"]
ci = pd.DataFrame({"estimate": point, "CI 2.5%": lo, "CI 97.5%": hi}, index=names).round(3)
display(ci)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for j, ax in enumerate(axes):
    ax.hist(boot[:, j], bins=40, color=BLUE, alpha=0.85)
    ax.axvline(lo[j], color=INK2, linestyle="--", linewidth=1.1)
    ax.axvline(hi[j], color=INK2, linestyle="--", linewidth=1.1)
    ax.axvline(point[j], color=ORANGE, linewidth=1.6)
    ax.set(title=names[j].replace(" district-days", "\ndistrict-days"), xlabel="Bootstrap estimate", ylabel="Count" if j == 0 else "")
save_show("04_bootstrap_ci")

**Findings.** (orange line = estimate, dashed lines = 95% interval)
* Mean daily rainfall: **4.28 mm** (95% CI ≈ 4.08 – 4.48).
* Share of district-days with rain: **44.4%** (≈ 43.2 – 45.6%).
* Heavy-or-above share: **0.68%** (≈ 0.59 – 0.78%).

**Why it matters.** The heavy-rain rate is known only to within about ±0.1 percentage points, and it rests on a handful of weather systems (EDA Chart 11). The intervals assume districts are independent samples; because 15 districts carry duplicated series (Module 2) and neighbouring districts share weather, the true uncertainty is probably a little wider. We report the intervals as a lower bound on uncertainty.

## Step 8 — Which candidate predictors relate to *tomorrow's* rainfall?
**What we are doing:** a preview of Module 4. For the 727 districts with complete records we compute candidate features **from information available today** and measure their Spearman correlation with **tomorrow's rainfall** (the thing we will forecast). Rolling features use only today and earlier days.

* `lag0` — today's rainfall; `mean3`, `mean7` — rolling mean of the last 3 / 7 days; `max7` — rolling maximum of the last 7 days;
* `dist_past` — the district's average rainfall so far (expanding mean up to today);
* `normal_next` — IMD's long-term *normal* for tomorrow (a climatological average, not derived from actual rainfall, so it does not leak);
* `day_of_year`, and `state_enc` as a plain number to illustrate the encoding issue.

In [ ]:
full_idx = grp.size()[lambda s: s == df["date"].nunique()].index
fd = df[df.set_index(["state", "district"]).index.isin(full_idx)].copy()
g2 = fd.groupby(["state", "district"])

fd["target_next_mm"] = g2["daily_actual"].shift(-1)
fd["normal_next"]    = g2["daily_normal"].shift(-1)
fd["lag0"]  = fd["daily_actual"]
fd["mean3"] = g2["daily_actual"].transform(lambda s: s.rolling(3).mean())
fd["mean7"] = g2["daily_actual"].transform(lambda s: s.rolling(7).mean())
fd["max7"]  = g2["daily_actual"].transform(lambda s: s.rolling(7).max())
fd["dist_past"] = g2["daily_actual"].transform(lambda s: s.expanding().mean())

cands = ["lag0", "mean3", "mean7", "max7", "dist_past", "normal_next", "day_of_year", "state_enc"]
t = fd.dropna(subset=["target_next_mm", "mean7", "normal_next"])
res = pd.Series({c: stats.spearmanr(t[c], t["target_next_mm"])[0] for c in cands}).sort_values()
print(f"Rows used: {len(t):,} (727 districts x 66 days: the first 6 days lack a full 7-day window and the last day has no next day)")
display(res.round(3).to_frame("Spearman with tomorrow's rain").iloc[::-1])

fig, ax = plt.subplots(figsize=(8, 4.3))
colors = [ORANGE if v < 0 else BLUE for v in res.values]
ax.barh(res.index, res.values, color=colors, height=0.6)
for i, v in enumerate(res.values):
    ax.text(v + (0.01 if v >= 0 else -0.01), i, f"{v:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9, color=INK2)
ax.axvline(0, color=INK2, linewidth=0.8)
ax.set(xlabel="Spearman correlation with next-day rainfall", title="Which signals relate to tomorrow's rain?")
ax.set_xlim(min(res.min() * 1.4, -0.1), res.max() * 1.2)
ax.grid(axis="y", visible=False)
save_show("05_predictors")

**Findings.**
* **Today's rainfall (0.57)** and the **3-day mean (0.51)** are the strongest signals. The 7-day mean (0.43) and the 7-day maximum (0.39) are weaker, so the very recent days matter most, matching the fast fade in Step 4.
* **IMD's normal for tomorrow (0.42)** is also informative. This looks higher than the 0.19 in Module 2's heatmap, but that was a *Pearson* correlation with the same day's rainfall; a rank correlation is much higher because the normal captures *where* and *when* it usually rains.
* `day_of_year` is **negatively** related (−0.34): rainfall falls as the season advances (Step 5).
* The district's running average (`dist_past`, 0.25) adds a moderate "this place is wet" signal.
* **`state_enc` is about 0.01, effectively zero.** That is not because state is unimportant (Step 6 shows it explains 51% of the variation); it is because `LabelEncoder` assigns numbers alphabetically, so "state 20" is not "more rainy" than "state 19".

**Decisions carried into Module 4.**
1. Build per-district features from the past only: `lag_1`, `lag_2`, `lag_3`, rolling mean 3 and 7, rolling max 7 (valid only where `days_since_prev_record` equals 1).
2. Use `day_of_year` rather than `month`; drop `year`.
3. Represent location with a **meaningful number**, such as the district's mean rainfall up to that day (`dist_past`), instead of the raw `state_enc` / `district_enc` integers. These integers are acceptable for tree models but meaningless for linear ones. We will test with and without them.
4. `daily_normal` of the target day is a candidate feature (it is climatology, not a function of actual rainfall), but Module 1's rule was written before this test. We will include it **only as a tested option** (with/without comparison in Module 5), and state this choice clearly in the report.
5. Split chronologically, because of the dependence shown in Step 4.

## Module 3 summary
| # | Question | Result |
|---|---|---|
| 2 | Typical values | Mean 4.28 mm, median 0, SD 11.4, 99th percentile 55 mm |
| 3 | Normal? | No (Shapiro W = 0.41); log helps skewness (5.7 → 1.4) but not the zero spike → use non-parametric tests |
| 4 | Does rain persist? | Lag-1 Spearman 0.57; P(rain tomorrow) 74% after rain vs 20% after dry; fades by lag 3 |
| 5 | Months differ? | Yes (Friedman p ≈ 1e-101, W ≈ 0.32); driven by October; Aug vs Sep not clearly different |
| 6 | States differ? | Yes (H ≈ 375); state explains ≈ 51% of district-level variation |
| 7 | Uncertainty | Mean 4.28 mm (4.08–4.48); rain share 44.4% (43.2–45.6); heavy+ 0.68% (0.59–0.78) |
| 8 | Predictors | lag0 0.57, mean3 0.51, normal_next 0.42, day_of_year −0.34; `state_enc` ≈ 0 as a raw integer |

**Limitations to report.** Only 73 days (no full seasonal cycle); strongly dependent observations (p-values are indicative, so effect sizes are reported); 15 districts with duplicated series and 4 all-zero districts (Module 2); no latitude/longitude, so spatial neighbour effects cannot be tested directly.

### Viva questions for Module 3
**Q1. Why did you use Spearman instead of Pearson?**
Rainfall is zero-inflated and extremely skewed, so Pearson is dominated by a few extreme days. Spearman uses ranks, assumes no distribution, and captures the common pattern ("rain follows rain").

**Q2. The normality tests reject almost everything with 53,102 rows. Is the p-value meaningful?**
With a huge sample even tiny departures are significant, so we rely on skewness (5.7), kurtosis and the Q–Q plot, which show a large, real departure caused by the zero spike.

**Q3. What does the 74% vs 20% result tell you?**
That days are not independent: rain is far more likely after rain. It supports lag features and forbids random splitting.

**Q4. Why a cluster bootstrap for the confidence intervals?**
Rows from one district are related, so treating 53,102 rows as independent would give intervals that are too narrow. Resampling whole districts respects that structure.

**Q5. Why is `state_enc` almost uncorrelated with rainfall although state explains 51% of the variance?**
`LabelEncoder` numbers are arbitrary labels (alphabetical order), not amounts. State matters, but the integer code does not carry that information to a linear model.

*Next: Module 4 — Feature Engineering (lags, rolling features, next-day target).*